In [1]:
library(sf)

Linking to GEOS 3.10.2, GDAL 3.4.1, PROJ 8.2.1; sf_use_s2() is TRUE



In [ ]:
## Load Required Dataframes
class = read.csv('Parcel_Classifications_2023.csv')
shape = st_read('Combined_Parcels_2024.shp')
wkbk = read.csv('Combined_Workbook_2023.csv')
wkbk2 = read.csv('Combined_Workbook_2024.csv')

Reading layer `Combined_Parcels_2024' from data source 
  `/anvil/projects/x-cis220051/sps/tippecanoe-project/Combined_Parcels_2024.shp' 
  using driver `ESRI Shapefile'


Warning message in CPL_read_ogr(dsn, layer, query, as.character(options), quiet, :
"GDAL Message 1: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection."
Warning message in CPL_read_ogr(dsn, layer, query, as.character(options), quiet, :
"GDAL Message 1: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection."
Warning message in CPL_read_ogr(dsn, layer, query, as.character(options), quiet, :
"GDAL Message 1: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection."
Warning message in CPL_read_ogr(dsn, layer, query, as.character(options), quiet, :
"GDAL Message 1: organizePolygons() recei

Simple feature collection with 3521572 features and 10 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 403530.9 ymin: 4180997 xmax: 692092.3 ymax: 4625472
Projected CRS: NAD83 / UTM zone 16N


In [3]:
library(ggplot2)
library(dplyr)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




In [4]:
wkbk = rbind(wkbk,wkbk2[wkbk2$County == 'Whitley',])
shape = inner_join(shape,wkbk,by=c('PrclNmb'='ParcelNumber'),relationship='many-to-many')

In [ ]:
## Make sure all Zip Codes are integers
shape$Zip = as.integer(shape$Zip)

Warning message:
"NAs introduced by coercion"


In [ ]:
## Map each Zip, County pair to an Urban/Rural classification
class = class %>%
group_by(Zip,County) %>%
summarise(Zip = first(Zip),County = first(County),ZipClassification = first(ZipClassification))
head(class)
nrow(class)

`summarise()` has grouped output by 'Zip'. You can override using the `.groups`
argument.


Zip,County,ZipClassification
<int>,<chr>,<chr>
40026,Clark,Urban
40059,Clark,Urban
40202,Clark,Urban
40212,Clark,Urban
40212,Floyd,Urban
40222,Clark,Urban


[1] 1275

In [14]:
class = inner_join(shape,class,by=c('Zip'='Zip','County.y'='County'),relationship='many-to-many')

In [ ]:
## Save each county's map of Zip Code classifications
class <- class %>%
  mutate(geometry = st_make_valid(geometry))
class$ZipClassification <- factor(class$ZipClassification, levels = c("Rural", "Urban"))
counties <- unique(class$County.y)

for (county in counties) {
  class_sub <- class[class$County.y == county, ]
  unioned <- class_sub %>%
    group_by(ZipClassification) %>%
    summarise(geometry = st_union(st_combine(geometry)), .groups = "drop")

  unioned$ZipClassification <- factor(unioned$ZipClassification, levels = c("Rural", "Urban"))

  p <- ggplot(unioned) + 
    geom_sf(aes(fill = ZipClassification), color = NA) + 
    scale_fill_discrete(drop = FALSE) +
    theme_minimal() +
    coord_sf(expand = FALSE) +
    labs(
      title = paste(county, "Zip Code Classification"),
      x = "Longitude", y = "Latitude"
    )

  ggsave(
    filename = paste0("county_pictures/zip_classification_", county, ".jpg"),
    plot = p,
    width = 8, height = 6,
    dpi = 300
  )
}